# Feed Monitor - View and Analyze Collected Posts

This notebook provides tools to monitor and analyze posts collected by the feed engine.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent.parent / "src"))

from feed.storage.neo4j_connection import get_connection
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)


## Connect to Neo4j


In [ ]:
neo4j = get_connection()
print(f"Connected to: {neo4j.uri}")


## Overview Statistics


In [ ]:
# Get total posts count
query = """
MATCH (p:Post)
RETURN count(p) as total_posts
"""
result = neo4j.execute_read(query)
total_posts = result[0]["total_posts"] if result else 0
print(f"Total Posts: {total_posts:,}")

# Get subreddits
query = """
MATCH (s:Subreddit)
OPTIONAL MATCH (s)<-[:POSTED_IN]-(p:Post)
RETURN s.name as subreddit, count(p) as post_count
ORDER BY post_count DESC
"""
result = neo4j.execute_read(query)
subreddits_df = pd.DataFrame([dict(record) for record in result])
print("\nPosts by Subreddit:")
print(subreddits_df.to_string(index=False))


## Duplicate Post Examples

Examples of duplicate posts (same content, different post IDs/subreddits):


In [ ]:
# Known duplicate examples
duplicate_examples = [
    {
        "group": 1,
        "posts": [
            "https://www.reddit.com/r/BrookeMonkTheSecond/comments/1psg76s/including_brooke_monk/",
            "https://www.reddit.com/r/BrookeMonkNSFWHub/comments/1pslxrj/including_brooke/",
        ]
    },
    {
        "group": 2,
        "posts": [
            "https://www.reddit.com/r/BrookeMonkNSFWHub/comments/1mvxr0a/so_perfect/",
            "https://www.reddit.com/r/BrookeMonkNSFWHub/comments/1nxmu0z/_/",
        ]
    }
]

def extract_post_id_from_url(url):
    """Extract Reddit post ID from URL."""
    import re
    match = re.search(r'/comments/([a-z0-9]+)/', url)
    return match.group(1) if match else None

def check_duplicates_in_db(duplicate_examples, neo4j):
    """Check if duplicate examples exist in database and show their details."""
    results = []
    
    for group in duplicate_examples:
        group_results = {"group": group["group"], "posts": []}
        
        for url in group["posts"]:
            post_id = extract_post_id_from_url(url)
            if not post_id:
                group_results["posts"].append({
                    "url": url,
                    "post_id": None,
                    "found": False,
                    "error": "Could not extract post ID"
                })
                continue
            
            query = """
            MATCH (p:Post {id: $post_id})
            OPTIONAL MATCH (p)-[:POSTED_IN]->(s:Subreddit)
            RETURN p.id as id,
                   p.title as title,
                   p.url as url,
                   p.permalink as permalink,
                   s.name as subreddit,
                   p.created_utc as created_utc
            """
            result = neo4j.execute_read(query, parameters={"post_id": post_id})
            
            if result:
                record = dict(result[0])
                group_results["posts"].append({
                    "url": url,
                    "post_id": post_id,
                    "found": True,
                    **record
                })
            else:
                group_results["posts"].append({
                    "url": url,
                    "post_id": post_id,
                    "found": False
                })
        
        results.append(group_results)
    
    return results

# Check duplicates
duplicate_results = check_duplicates_in_db(duplicate_examples, neo4j)

for group_result in duplicate_results:
    print(f"\n=== Duplicate Group {group_result['group']} ===")
    for post in group_result["posts"]:
        if post["found"]:
            print(f"  Found: {post['post_id']}")
            print(f"    Title: {post.get('title', 'N/A')}")
            print(f"    Subreddit: {post.get('subreddit', 'N/A')}")
            print(f"    Image URL: {post.get('url', 'N/A')}")
            print(f"    Permalink: {post.get('permalink', 'N/A')}")
        else:
            print(f"  Not found: {post['post_id']} ({post['url']})")


## Find Duplicates by Image URL

Since duplicate posts often share the same image URL, we can find duplicates by grouping posts by their `url` field (the image URL).


In [ ]:
# Find posts that share the same image URL (potential duplicates)
query = """
MATCH (p:Post)
WHERE p.url IS NOT NULL AND p.url <> ""
WITH p.url as image_url, collect(p) as posts
WHERE size(posts) > 1
RETURN image_url,
       size(posts) as duplicate_count,
       [post IN posts | {
           id: post.id,
           title: post.title,
           subreddit: [(post)-[:POSTED_IN]->(s) | s.name][0],
           permalink: post.permalink,
           created_utc: post.created_utc
       }] as duplicate_posts
ORDER BY duplicate_count DESC
LIMIT 20
"""

result = neo4j.execute_read(query)
duplicates_df = pd.DataFrame([dict(record) for record in result])

if not duplicates_df.empty:
    print(f"Found {len(duplicates_df)} image URLs with duplicates")
    print(f"Total duplicate groups: {duplicates_df['duplicate_count'].sum() - len(duplicates_df)}")
    print("\nTop duplicate groups:")
    for idx, row in duplicates_df.head(10).iterrows():
        print(f"\nImage URL: {row['image_url']}")
        print(f"  Appears in {row['duplicate_count']} posts:")
        for post in row['duplicate_posts']:
            print(f"    - {post['id']}: {post['title'][:60]}...")
            print(f"      r/{post['subreddit']} - {post['permalink']}")
else:
    print("No duplicates found by image URL")


## Find Duplicates by Image Hash

Find posts that share the same image hash (exact duplicate images, even if URLs differ):


In [ ]:
# Find posts with duplicate image hashes
query = """
MATCH (p:Post)
WHERE p.image_hash IS NOT NULL AND p.image_hash <> ""
WITH p.image_hash as image_hash, collect(p) as posts
WHERE size(posts) > 1
RETURN image_hash,
       size(posts) as duplicate_count,
       [post IN posts | {
           id: post.id,
           title: post.title,
           subreddit: [(post)-[:POSTED_IN]->(s) | s.name][0],
           permalink: post.permalink,
           url: post.url,
           created_utc: post.created_utc
       }] as duplicate_posts
ORDER BY duplicate_count DESC
LIMIT 20
"""

result = neo4j.execute_read(query)
hash_duplicates_df = pd.DataFrame([dict(record) for record in result])

if not hash_duplicates_df.empty:
    print(f"Found {len(hash_duplicates_df)} image hashes with duplicates")
    print(f"Total duplicate groups: {hash_duplicates_df['duplicate_count'].sum() - len(hash_duplicates_df)}")
    print("\nTop duplicate groups by image hash:")
    for idx, row in hash_duplicates_df.head(10).iterrows():
        print(f"\nImage Hash: {row['image_hash'][:16]}...")
        print(f"  Appears in {row['duplicate_count']} posts:")
        for post in row['duplicate_posts']:
            print(f"    - {post['id']}: {post['title'][:60]}...")
            print(f"      r/{post['subreddit']} - {post['url']}")
            print(f"      {post['permalink']}")
else:
    print("No duplicates found by image hash")
    print("(Note: Image hashing may not be enabled, or no duplicates exist yet)")
